In [2]:
%matplotlib inline

import matplotlib

print(matplotlib.get_backend())

module://matplotlib_inline.backend_inline


# Parsing TYNDP data for SHRECC

In [3]:
import pandas as pd
import numpy as np
import scipy as sp
import pickle
import os
import plotly.express as px

%load_ext jupyter_black

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black


In [4]:
matplotlib

<module 'matplotlib' from 'd:\\.conda\\envs\\shrecc\\Lib\\site-packages\\matplotlib\\__init__.py'>

In [5]:
pickle

<module 'pickle' from 'd:\\.conda\\envs\\shrecc\\Lib\\pickle.py'>

# Import TYNDP data

In [6]:
filepath_xlsb = "../data/tyndp/MMStandardOutputFile_DE2050_Plexos_CY2009_v11_SoS.xlsb"
filepath_prod_pkl = "../data/tyndp/DE2050_CY2009_prod.pkl"
filepath_trade_pkl = "../data/tyndp/DE2050_CY2009_trade.pkl"
filepath_concordance_techs = "../data/tyndp/tyndp_ei_mapping.xlsx"
filepath_concordance_nodes = "../data/tyndp/tyndp_country_mapping.xlsx"

In [7]:
if os.path.isfile(filepath_prod_pkl):
    with open(filepath_prod_pkl, "rb") as f:
        prod_DE2050_CY2009 = pickle.load(f)
else:
    prod_DE2050_CY2009 = pd.read_excel(
        io=filepath_xlsb,
        sheet_name="Hourly Market Data emarket",
        header=[10, 11, 12],
        index_col=[0, 1],
        engine="pyxlsb",
    )
    with open(filepath_prod_pkl, "wb") as f:
        pickle.dump(prod_DE2050_CY2009, f)

if os.path.isfile(filepath_trade_pkl):
    with open(filepath_trade_pkl, "rb") as f:
        trade_DE2050_CY2009 = pickle.load(f)
else:
    trade_DE2050_CY2009 = pd.read_excel(
        io=filepath_xlsb,
        sheet_name="Crossborder exchanges",
        header=[10],
        index_col=[0, 1],
        engine="pyxlsb",
    )
    with open(filepath_trade_pkl, "wb") as f:
        pickle.dump(trade_DE2050_CY2009, f)

## Production

In [8]:
# This is the raw data from TYNDP
# Parsing and cleaning includes:
# - keep only the columns with production data for nodes XX00, XX00RETE
# - remove columns where all values are NaN
# - set a proper datetime index

prod_DE2050_CY2009.head()

,Category,Nuclear [MW],Lignite old 1 [MW],Lignite old 2 [MW],Lignite new [MW],Lignite CCS [MW],Hard coal old 1 [MW],Hard coal old 2 [MW],Hard coal new [MW],Hard coal CCS [MW],Gas conventional old 1 [MW],...,Hydrogen CCGT [MW],Demand Side Response [MW],CH4 Heat Pump (load) [MW],H2 Heat Pump (load) [MW],Adequacy [MW],Demand [MW]\n(losses included),Balance [MW],Dumped Energy [MW],Energy Not Served [MW],Marginal Cost [€]
,Country,AL00,AL00,AL00,AL00,AL00,AL00,AL00,AL00,AL00,AL00,...,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE,UKNIRETE
,Date /Code,AL00_1,AL00_2,AL00_3,AL00_4,AL00_5,AL00_6,AL00_7,AL00_8,AL00_9,AL00_10,...,UKNIRETE_52,UKNIRETE_DSR,UKNIRETE_HCH4,UKNIRETE_HH2,UKNIRETE_SoS,UKNIRETE_LOAD,UKNIRETE_Balance,UKNIRETE_Dump,UKNIRETE_ENS,UKNIRETE_Mgl Cost
1,01JAN00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2214,-2214,0,0,237
2,01JAN01:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2206,-2206,0,0,233
3,01JAN02:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2177,-2177,0,0,231
4,01JAN03:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2173,-2173,0,0,231
5,01JAN04:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0,0,NaN,2150,-2150,0,0,231


In [9]:
# index in the proper format
dt = pd.to_datetime(
    prod_DE2050_CY2009.index.get_level_values(1) + "2050", format="%d%b%H:%M%Y"
)

In [10]:
# Let's have a look at production data first
prod_DE2050_CY2009.dropna(axis=1, how="all", inplace=True)
prod_DE2050_CY2009.index = dt
prod_DE2050_CY2009.head()

Category            Gas CCGT old 2 [MW] Gas CCGT new [MW] Run-of-River [MW]  \
Country                            AL00              AL00              AL00   
Date /Code                      AL00_13           AL00_14           AL00_26   
2050-01-01 00:00:00                   0               110               193   
2050-01-01 01:00:00                   0               100               193   
2050-01-01 02:00:00                   0                90               193   
2050-01-01 03:00:00                   0                 0               193   
2050-01-01 04:00:00                   0                 0               193   

Category            Reservoir [MW] Wind Onshore [MW]  \
Country                       AL00              AL00   
Date /Code                 AL00_27           AL00_31   
2050-01-01 00:00:00           1904                40   
2050-01-01 01:00:00           1904                43   
2050-01-01 02:00:00           1904                48   
2050-01-01 03:00:00           1904                55   
2050-01-01 04:00:00           1904                59   

Category            Solar (Photovoltaic) [MW] Demand [MW]\n(losses included)  \
Country                                  AL00                           AL00   
Date /Code                            AL00_33                      AL00_LOAD   
2050-01-01 00:00:00                         0                           1145   
2050-01-01 01:00:00                         0                           1006   
2050-01-01 02:00:00                         0                            923   
2050-01-01 03:00:00                         0                            885   
2050-01-01 04:00:00                         0                            892   

Category            Balance [MW] Dumped Energy [MW] Energy Not Served [MW]  \
Country                     AL00               AL00                   AL00   
Date /Code          AL00_Balance          AL00_Dump               AL00_ENS   
2050-01-01 00:00:00         1102                  0                      0   
2050-01-01 01:00:00         1234                  0                      0   
2050-01-01 02:00:00         1312                  0                      0   
2050-01-01 03:00:00         1267                  0                      0   
2050-01-01 04:00:00         1265                  0                      0   

Category             ...                Marginal Cost [€]  \
Country              ...     UKNI SRES          UKNI SRES   
Date /Code           ... UKNI SRES_ENS UKNI SRES_Mgl Cost   
2050-01-01 00:00:00  ...             0              -1000   
2050-01-01 01:00:00  ...             0              -1000   
2050-01-01 02:00:00  ...             0              -1000   
2050-01-01 03:00:00  ...             0              -1000   
2050-01-01 04:00:00  ...             0              -1000   

Category            Solar (Photovoltaic) [MW] CH4 Heat Pump (load) [MW]  \
Country                              UKNIRETE                  UKNIRETE   
Date /Code                        UKNIRETE_33             UKNIRETE_HCH4   
2050-01-01 00:00:00                         0                         0   
2050-01-01 01:00:00                         0                         0   
2050-01-01 02:00:00                         0                         0   
2050-01-01 03:00:00                         0                         0   
2050-01-01 04:00:00                         0                         0   

Category            H2 Heat Pump (load) [MW] Demand [MW]\n(losses included)  \
Country                             UKNIRETE                       UKNIRETE   
Date /Code                      UKNIRETE_HH2                  UKNIRETE_LOAD   
2050-01-01 00:00:00                        0                           2214   
2050-01-01 01:00:00                        0                           2206   
2050-01-01 02:00:00                        0                           2177   
2050-01-01 03:00:00                        0                           2173   
2050-01-0

In [11]:
test = prod_DE2050_CY2009.xs("AL00", level=1, axis=1)

test.columns

MultiIndex([(           'Gas CCGT old 2 [MW]',       'AL00_13'),
            (             'Gas CCGT new [MW]',       'AL00_14'),
            (             'Run-of-River [MW]',       'AL00_26'),
            (                'Reservoir [MW]',       'AL00_27'),
            (             'Wind Onshore [MW]',       'AL00_31'),
            (     'Solar (Photovoltaic) [MW]',       'AL00_33'),
            ('Demand [MW]\n(losses included)',     'AL00_LOAD'),
            (                  'Balance [MW]',  'AL00_Balance'),
            (            'Dumped Energy [MW]',     'AL00_Dump'),
            (        'Energy Not Served [MW]',      'AL00_ENS'),
            (             'Marginal Cost [€]', 'AL00_Mgl Cost')],
           names=['Category', 'Date /Code'])

In [12]:
test.loc["2050-01-01 00:00:00"]

Category                        Date /Code   
Gas CCGT old 2 [MW]             AL00_13             0
Gas CCGT new [MW]               AL00_14           110
Run-of-River [MW]               AL00_26           193
Reservoir [MW]                  AL00_27          1904
Wind Onshore [MW]               AL00_31            40
Solar (Photovoltaic) [MW]       AL00_33             0
Demand [MW]\n(losses included)  AL00_LOAD        1145
Balance [MW]                    AL00_Balance     1102
Dumped Energy [MW]              AL00_Dump           0
Energy Not Served [MW]          AL00_ENS            0
Marginal Cost [€]               AL00_Mgl Cost     115
Name: 2050-01-01 00:00:00, dtype: int64

In [13]:
# Production should equal Demand + Commercial balance

In [14]:
test.iloc[:, :6].sum(1) - test[["Demand [MW]\n(losses included)", "Balance [MW]"]].sum(
    1
)

2050-01-01 00:00:00    0
2050-01-01 01:00:00    0
2050-01-01 02:00:00    0
2050-01-01 03:00:00    0
2050-01-01 04:00:00   -1
                      ..
2050-12-30 19:00:00    1
2050-12-30 20:00:00    0
2050-12-30 21:00:00    0
2050-12-30 22:00:00    0
2050-12-30 23:00:00    1
Length: 8736, dtype: int64

### Technologies

In [15]:
# Isolate sources
sources = prod_DE2050_CY2009.columns.levels[0]

In [16]:
concordance = pd.read_excel(
    filepath_concordance_techs, sheet_name="concordance", index_col=0, skipfooter=1
).iloc[:, :-1]
concordance = concordance.loc[concordance.sum(1) > 0, concordance.sum() > 0]
concordance /= concordance.sum()
concordance.columns.name = "Category"
concordance.head()

Category,Gas CCGT CCS [MW],Gas CCGT new [MW],Gas CCGT old 1 [MW],Gas CCGT old 2 [MW],Gas CCGT present 1 [MW],Gas CCGT present 2 [MW],Gas OCGT new [MW],Gas OCGT old [MW],Gas conventional old 1 [MW],Gas conventional old 2 [MW],...,Others non-renewable [MW],Others renewable [MW],Pump Storage - Closed Loop (turbine) [MW],Pump Storage - Open Loop (turbine) [MW],Reservoir [MW],Run-of-River [MW],Solar (Photovoltaic) [MW],Solar (Thermal) [MW],Wind Offshore [MW],Wind Onshore [MW]
premise activity,,,,,,,,,,,,,,,,,,,,,
"electricity production, at biomass-fired IGCC power plant, pre, pipeline 200km, storage 1000m",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at co-generation natural gas-fired power plant, post, pipeline 200km, storage 1000m",0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at co-generation oil-fired power plant, post, pipeline 200km, storage 1000m",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at co-generation wood-fired power plant, post, pipeline 200km, storage 1000m",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"electricity production, at hard coal-fired IGCC power plant",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Nodes

In [17]:
country = "LU"
index_c = [c for c in prod_DE2050_CY2009.columns.levels[1] if c.startswith(country)]
index_c

['LUB1',
 'LUB1 SRES',
 'LUF1',
 'LUF1 SRES',
 'LUG1',
 'LUG1 EV Passenger Prosumer',
 'LUG1 EV Passenger Street',
 'LUG1 SRES',
 'LUG1RETE',
 'LUV1',
 'LUV1 SRES']

In [ ]:
idx = pd.IndexSlice

prod_DE2050_CY2009.loc[:, idx[:, index_c]].dropna(how="all", axis=1).sum().sort_values()

Category                        Country                     Date /Code                         
Marginal Cost [€]               LUV1 SRES                   LUV1 SRES_Mgl Cost                    -8736000
                                LUF1 SRES                   LUF1 SRES_Mgl Cost                    -8736000
                                LUB1 SRES                   LUB1 SRES_Mgl Cost                    -8736000
                                LUG1 SRES                   LUG1 SRES_Mgl Cost                    -8736000
Balance [MW]                    LUG1                        LUG1_Balance                          -3647483
                                                                                                    ...   
Solar (Photovoltaic) [MW]       LUG1RETE                    LUG1RETE_33                            1711841
Demand [MW]\n(losses included)  LUG1 EV Passenger Prosumer  LUG1 EV Passenger Prosumer_LOAD        1958164
Marginal Cost [€]               LUG1 EV Passenge

: 

In [ ]:
month = 1
prod_DE2050_CY2009.loc[:, idx[:, index_c]].iloc[
    month * 30 * 24 : (month * 30 + 7) * 24, :
].plot.area()

# sample = sample.loc[:, sample.sum() > 0]


# ax = sample.plot.area(figsize=(10, 6))

ValueError: When stacked is True, each column must be either all positive or all negative. Column '(Balance [MW], LUG1 EV Passenger Prosumer, LUG1 EV Passenger Prosumer_Balance)' contains both positive and negative values

In [ ]:
# sample.index = pd.to_datetime(sample.index, format="%d%b%H:%M").map(
#     lambda x: x.replace(year=2050)
# )

In [17]:
# plot_sample = sample.iloc[:, :10] if sample.shape[1] > 10 else sample
# fig = go.Figure()
# for col in plot_sample.columns:
#     fig.add_trace(
#         go.Scatter(
#             x=plot_sample.index,
#             y=plot_sample[col],
#             name=str(col),
#             mode="lines",
#             stackgroup="one",
#         )
#     )
# fig.update_layout(
#     title="TYNDP sample generation",
#     xaxis_title="Time",
#     yaxis_title="Generation (MW)",
#     hovermode="x unified",
#     height=600,
#     width=1000,
# )
# fig.show()

In [18]:
prod_category_mask = prod_DE2050_CY2009.columns.get_level_values("Category").isin(
    concordance.columns
)
prod_DE2050_CY2009 = prod_DE2050_CY2009.loc[:, prod_category_mask]
countries = prod_DE2050_CY2009.columns.get_level_values("Country").unique()

In [19]:
sp.linalg.block_diag(*[concordance.to_numpy()] * len(countries)).shape

(8436, 7548)

In [20]:
concordance_full = pd.DataFrame(
    sp.linalg.block_diag(*[concordance.to_numpy()] * len(countries)),
    index=pd.MultiIndex.from_product([countries, concordance.index]),
    columns=pd.MultiIndex.from_product([countries, concordance.columns]),
)

In [21]:
prod_DE2050_CY2009.columns.droplevel("Date /Code")

MultiIndex([(           'Gas CCGT old 2 [MW]',      'AL00'),
            (             'Gas CCGT new [MW]',      'AL00'),
            (             'Run-of-River [MW]',      'AL00'),
            (                'Reservoir [MW]',      'AL00'),
            (             'Wind Onshore [MW]',      'AL00'),
            (     'Solar (Photovoltaic) [MW]',      'AL00'),
            ('Demand [MW]\n(losses included)',      'AL00'),
            (                  'Balance [MW]',      'AL00'),
            (            'Dumped Energy [MW]',      'AL00'),
            (        'Energy Not Served [MW]',      'AL00'),
            ...
            (        'Energy Not Served [MW]', 'UKNI SRES'),
            (             'Marginal Cost [€]', 'UKNI SRES'),
            (     'Solar (Photovoltaic) [MW]',  'UKNIRETE'),
            (     'CH4 Heat Pump (load) [MW]',  'UKNIRETE'),
            (      'H2 Heat Pump (load) [MW]',  'UKNIRETE'),
            ('Demand [MW]\n(losses included)',  'UKNIRETE'),
        

In [22]:
concordance_full.columns

MultiIndex([(    'AL00',                         'Gas CCGT CCS [MW]'),
            (    'AL00',                         'Gas CCGT new [MW]'),
            (    'AL00',                       'Gas CCGT old 1 [MW]'),
            (    'AL00',                       'Gas CCGT old 2 [MW]'),
            (    'AL00',                   'Gas CCGT present 1 [MW]'),
            (    'AL00',                   'Gas CCGT present 2 [MW]'),
            (    'AL00',                         'Gas OCGT new [MW]'),
            (    'AL00',                         'Gas OCGT old [MW]'),
            (    'AL00',               'Gas conventional old 1 [MW]'),
            (    'AL00',               'Gas conventional old 2 [MW]'),
            ...
            ('UKNIRETE',                 'Others non-renewable [MW]'),
            ('UKNIRETE',                     'Others renewable [MW]'),
            ('UKNIRETE', 'Pump Storage - Closed Loop (turbine) [MW]'),
            ('UKNIRETE',   'Pump Storage - Open Loop (turbine

In [24]:
concordance_like_prod = concordance_full.reindex(
    prod_DE2050_CY2009.columns.droplevel("Date /Code").swaplevel(), axis=1
).fillna(0)
concordance_like_prod.columns = prod_DE2050_CY2009.columns
concordance_like_prod.head(2)

Category                                                   Gas CCGT old 2 [MW]  \
Country                                                                   AL00   
Date /Code                                                             AL00_13   
Country premise activity                                                         
AL00    electricity production, at biomass-fired IGCC p...                 0.0   
        electricity production, at co-generation natura...                 0.0   

Category                                                   Gas CCGT new [MW]  \
Country                                                                 AL00   
Date /Code                                                           AL00_14   
Country premise activity                                                       
AL00    electricity production, at biomass-fired IGCC p...               0.0   
        electricity production, at co-generation natura...               0.0   

Category                                                   Run-of-River [MW]  \
Country                                                                 AL00   
Date /Code                                                           AL00_26   
Country premise activity                                                       
AL00    electricity production, at biomass-fired IGCC p...               0.0   
        electricity production, at co-generation natura...               0.0   

Category                                                   Reservoir [MW]  \
Country                                                              AL00   
Date /Code                                                        AL00_27   
Country premise activity                                                    
AL00    electricity production, at biomass-fired IGCC p...            0.0   
        electricity production, at co-generation natura...            0.0   

Category                                                   Wind Onshore [MW]  \
Country                                                                 AL00   
Date /Code                                                           AL00_31   
Country premise activity                                                       
AL00    electricity production, at biomass-fired IGCC p...               0.0   
        electricity production, at co-generation natura...               0.0   

Category                                                   Solar (Photovoltaic) [MW]  \
Country                                                                         AL00   
Date /Code                                                                   AL00_33   
Country premise activity                                                               
AL00    electricity production, at biomass-fired IGCC p...                       0.0   
        electricity production, at co-generation natura...                       0.0   

Category                                                   Demand [MW]\n(losses included)  \
Country                                                                              AL00   
Date /Code                                                                      AL00_LOAD   
Country premise activity                                                                    
AL00    electricity production, at biomass-fired IGCC p...                            0.0   
        electricity production, at co-generation natura...                            0.0   

Category                                                   Balance [MW]  \
Country                                                            AL00   
Date /Code                                                 AL00_Balance   
Country premise activity                                                  
AL00    electricity production, at biomass-fired IGCC p...          0.0   
        electricity production, at co-generation natura...          0.0   

Category                                                   Dum

In [25]:
prod_DE2050_CY2009.shape

(8736, 2077)

In [26]:
concordance_like_prod.shape

(8436, 2077)

In [27]:
prod_DE2050_CY2009.columns.equals(concordance_like_prod.columns)

True

In [28]:
prod_values = prod_DE2050_CY2009.to_numpy(dtype="float64", copy=False)
conc_sparse = sp.sparse.csr_matrix(
    concordance_like_prod.to_numpy(dtype="float64", copy=False)
)
prod_agg_values = prod_values @ conc_sparse.T

In [29]:
prod_agg_ei = pd.DataFrame(
    prod_agg_values, index=prod_DE2050_CY2009.index, columns=concordance_like_prod.index
)

In [30]:
prod_agg_ei.loc["2050-01-01 00:00:00"].nlargest(30)

Country  premise activity                                                                                              
FI00     electricity production, wind, <1MW turbine, onshore                                                               38852.0
         electricity production, wind, >3MW turbine, onshore                                                               38852.0
DE00     electricity production, at hydrogen-fired combined cycle power plant, by auto-thermal reforming of natural gas    20436.0
NL00     electricity production, at hydrogen-fired combined cycle power plant, by auto-thermal reforming of natural gas    14341.0
FR00     electricity production, nuclear, boiling water reactor                                                            11537.0
         electricity production, nuclear, pressure water reactor                                                           11537.0
NOS0     electricity production, hydro, reservoir, alpine region                              

In [32]:
concordance_nodes = pd.read_excel(
    filepath_concordance_nodes, sheet_name="countries", index_col=0
)
concordance_nodes["Country code"].to_dict()

{'AL00': 'AL',
 'AT00': 'AT',
 'BA00': 'BA',
 'BE00': 'BE',
 'BG00': 'BG',
 'EEOF': 'BW',
 'CH00': 'CH',
 'CW00': 'CW',
 'CY00': 'CY',
 'CZ00': 'CZ',
 'DE00': 'DE',
 'DEKF': 'DE',
 'DKBH': 'DK',
 'DKE1': 'DK',
 'DKKF': 'DK',
 'DKW1': 'DK',
 'DZ00': 'DZ',
 'EE00': 'EE',
 'EG00': 'EG',
 'ES00': 'ES',
 'FI00': 'FI',
 'FR00': 'FR',
 'FR15': 'FR',
 'GR00': 'GR',
 'GR03': 'GR',
 'HR00': 'HR',
 'HU00': 'HU',
 'IE00': 'IE',
 'IL00': 'IL',
 'ITCA': 'IT',
 'ITCN': 'IT',
 'ITCO': 'IT',
 'ITCS': 'IT',
 'ITN1': 'IT',
 'ITS1': 'IT',
 'ITSA': 'IT',
 'ITSI': 'IT',
 'ITVI': 'IT',
 'LT00': 'LT',
 'LTOF': 'LT',
 'LUB1': 'LU',
 'LUF1': 'LU',
 'LUG1': 'LU',
 'LUV1': 'LU',
 'LV00': 'LV',
 'LY00': 'LY',
 'MA00': 'MA',
 'MD00': 'MD',
 'ME00': 'ME',
 'MK00': 'MK',
 'MT00': 'MT',
 'NL00': 'NL',
 'NL6H': 'NL',
 'NLA0': 'NL',
 'NLBH': 'NL',
 'NLLL': 'NL',
 'NOM1': 'NO',
 'NON1': 'NO',
 'NOS0': 'NO',
 'BEOF': 'NW',
 'DKNS': 'NW',
 'NL60': 'NW',
 'PL00': 'PL',
 'PLE0': 'PL',
 'PLI0': 'PL',
 'PS00': 'PS',
 'PT00': '

In [33]:
prod_agg_ei.shape

(8736, 8436)

In [36]:
prod_agg_ei = (
    prod_agg_ei.rename(
        concordance_nodes["Country code"].to_dict(), level="Country", axis=1
    )
    .groupby(["Country", "premise activity"], axis=1)
    .sum()
)

C:\Users\Gibon\AppData\Local\Temp\ipykernel_26860\3370550903.py:5: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  .groupby(["Country", "premise activity"], axis=1)


## Trade

Trade data requires special attention. Nodes are treated separately (this is OK), but it looks like various interconnection alternatives exist (Concept 1/Concept 2), this is unclear.

In [38]:
# same for trade data
trade_DE2050_CY2009.dropna(axis=1, how="all", inplace=True)
trade_DE2050_CY2009.index = dt
trade_DE2050_CY2009.head()

,AL00-GR00,AL00-GR00 Concept 1,AL00-GR00 Concept 2,AL00-ME00,AL00-ME00 Concept 1,AL00-ME00 Concept 2,AL00-ME00 Concept 3,AL00-ME00 Concept 4,AL00-MK00,AL00-MK00 Concept 1,...,UKNIEV-UKNIRETE,UKNIRETE-UKNIEV,UKNI-UKNIEV,UKNI-UKNIRETE,UKOH001OHEL-UK00 EXP,UKOH003OHEL-UK00 EXP,UKOH004OHEL-IE00 EXP,UKOH004OHEL-UK00 EXP,UKOH005OHEL-UK00 EXP,UKOH006OHEL-UK00 EXP
2050-01-01 00:00:00,-600,-500,-500,330,468,468,468,424,-330,-330,...,0,0,0,2214,-3877,-2377,-2687,5610,8276,6526
2050-01-01 01:00:00,-600,-500,-500,298,439,439,439,383,-185,-185,...,0,0,0,2206,-3877,-4437,-2574,5610,8261,6397
2050-01-01 02:00:00,-600,-500,-500,314,456,456,456,405,-191,-191,...,0,2,1,2179,-3877,-4844,-2489,5610,8201,6139
2050-01-01 03:00:00,-600,-500,-500,323,463,463,463,416,-234,-234,...,0,2,1,2175,-3877,-4627,-2291,5610,8177,6053
2050-01-01 04:00:00,-600,-500,-500,329,467,467,467,422,-247,-247,...,0,1,1,2152,-3877,-4459,-2093,5610,8226,5967


In [39]:
# We want to reshape the trade data with origin-destination pairs as multiindex columns
parsed = trade_DE2050_CY2009.columns.to_series().str.extract(
    r"^\s*(?P<origin>.+?)\s*-\s*(?P<destination>.+?)\s*$"
)

# If any origin/destination is NaN, raise a flag
bad_cols = trade_DE2050_CY2009.columns[
    parsed["origin"].isna() | parsed["destination"].isna()
]
print(bad_cols)

# Clean the names by removing trailing "Concept N" and stripping whitespace
parsed["origin"] = (
    parsed["origin"].str.replace(r"\s*/?Concept\s+\d+\s*$", "", regex=True).str.strip()
)

parsed["destination"] = (
    parsed["destination"]
    .str.replace(r"\s*/?Concept\s+\d+\s*$", "", regex=True)
    .str.strip()
)

# Apply the cleaned names to the columns
trade = trade_DE2050_CY2009.copy()
trade.columns = pd.MultiIndex.from_arrays(
    [parsed["origin"], parsed["destination"]], names=["origin", "destination"]
)

# Origin-destination node pairs for each hour
od_hourly = trade.T.groupby(level=["origin", "destination"]).sum().T

Index([], dtype='object')


In [40]:
od_hourly.head()

origin               AL00                    AT00                             \
destination          GR00  ME00 MK00  RS00 AT00EV AT00RETE  CH00  CZ00  DE00   
2050-01-01 00:00:00 -1600  2158 -705  1250      0     7605 -1700  2400  7147   
2050-01-01 01:00:00 -1600  1998 -414  1250      0     7465 -1700  2400  6801   
2050-01-01 02:00:00 -1600  2087 -426  1250      0     7358 -1700  2400  6510   
2050-01-01 03:00:00 -1600  2128 -512  1250      0     7266 -1700  2400  6702   
2050-01-01 04:00:00 -1600  2152 -538  1250      0     7192 -1700  2400  6902   

origin                     ...     UKNI UKNIEV          UKNIRETE UKOH001OHEL  \
destination          HU00  ... UKNIRETE   UKNI UKNIRETE   UKNIEV    UK00 EXP   
2050-01-01 00:00:00 -1124  ...     2214      0        0        0       -3877   
2050-01-01 01:00:00 -1262  ...     2206      0        0        0       -3877   
2050-01-01 02:00:00 -1336  ...     2179      0        0        2       -3877   
2050-01-01 03:00:00 -1343  ...     2175      0        0        2       -3877   
2050-01-01 04:00:00 -1258  ...     2152      0        0        1       -3877   

origin              UKOH003OHEL UKOH004OHEL          UKOH005OHEL UKOH006OHEL  
destination            UK00 EXP    IE00 EXP UK00 EXP    UK00 EXP    UK00 EXP  
2050-01-01 00:00:00       -2377       -2687     5610        8276        6526  
2050-01-01 01:00:00       -4437       -2574     5610        8261        6397  
2050-01-01 02:00:00       -4844       -2489     5610        8201        6139  
2050-01-01 03:00:00       -4627       -2291     5610        8177        6053  
2050-01-01 04:00:00       -4459       -2093     5610        8226        5967  

[5 rows x 388 columns]

In [41]:
lu_imports = od_hourly.columns.get_level_values("destination").str.startswith(
    "LU"
) & ~od_hourly.columns.get_level_values("origin").str.startswith("LU")

lu_exports = od_hourly.columns.get_level_values("origin").str.startswith("LU")


print(od_hourly.loc[:, lu_imports].head())
print(od_hourly.loc[:, lu_exports].head())

origin              BE00                  DE00      FR00
destination         LUB1 LUG1 LUG1 Real 1 LUG1 LUV1 LUF1
2050-01-01 00:00:00   27  800         500  374    0  121
2050-01-01 01:00:00   25  800         500  332    0  134
2050-01-01 02:00:00   26  800         500  316    0  125
2050-01-01 03:00:00   25  800         500  321    0  130
2050-01-01 04:00:00   25  800         500  313    0  113
origin                LUG1          LUG1EV          LUG1RETE
destination         LUG1EV LUG1RETE   LUG1 LUG1RETE   LUG1EV
2050-01-01 00:00:00    228     1098      0        0      327
2050-01-01 01:00:00    228     1126      0        0      367
2050-01-01 02:00:00    227     1150      0        0      387
2050-01-01 03:00:00    222     1142      0        0      375
2050-01-01 04:00:00    218     1129      0        0      351


In [42]:
od_hourly.loc[:, lu_imports].sum()

origin  destination
BE00    LUB1            248394
        LUG1           1926318
        LUG1 Real 1    1051391
DE00    LUG1           6219713
        LUV1            343015
FR00    LUF1           1470076
dtype: int64

In [43]:
px.area(od_hourly.loc[:, lu_imports].groupby(level="origin", axis=1).sum())

C:\Users\Gibon\AppData\Local\Temp\ipykernel_26860\4228155494.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  px.area(od_hourly.loc[:, lu_imports].groupby(level="origin", axis=1).sum())


In [44]:
# Let's use Ember mapping to aggregate nodes

concordance_connections = pd.read_excel(
    filepath_concordance_nodes, sheet_name="connections", index_col=0
)
concordance_connections = concordance_connections[
    ~concordance_connections.index.duplicated(keep="first")
]

In [45]:
trade_DE2050_CY2009.columns.name = "BZ line"

In [46]:
trade_DE2050_CY2009.columns = pd.MultiIndex.from_frame(
    concordance_connections.reindex(trade_DE2050_CY2009.columns, axis=0).reset_index()
)

In [47]:
trade_agg = (
    trade_DE2050_CY2009.groupby(level=["Country from", "Country to"], axis=1)
    .sum()
    .head()
)

C:\Users\Gibon\AppData\Local\Temp\ipykernel_26860\1912001581.py:2: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  trade_DE2050_CY2009.groupby(level=["Country from", "Country to"], axis=1)


In [48]:
trade_agg

Country from          AL                   AT                            ...  \
Country to            GR   ME   MK   RS    CH   CZ    DE   HU   IT   SI  ...   
2050-01-01 00:00:00 -600  330 -330  250 -1200  900  7147 -500 -695 -950  ...   
2050-01-01 01:00:00 -600  298 -185  250 -1200  900  6801 -555 -695 -950  ...   
2050-01-01 02:00:00 -600  314 -191  250 -1200  900  6510 -586 -695 -950  ...   
2050-01-01 03:00:00 -600  323 -234  250 -1200  900  6702 -589 -695 -950  ...   
2050-01-01 04:00:00 -600  329 -247  250 -1200  900  6902 -554 -695 -950  ...   

Country from          LT             ME   MK   NL         NO   PL    RO  
Country to            LV   PL   SE   RS   RS   NO    UK   SE   SE    RS  
2050-01-01 00:00:00    0  700  700  540  460 -700 -1000 -418 -600  1984  
2050-01-01 01:00:00    0  700  354  540  460 -700 -1000 -726 -600  1956  
2050-01-01 02:00:00 -747  700   49  540  460 -700 -1000 -856 -600  1426  
2050-01-01 03:00:00 -950  700  -80  540  460 -700 -1000 -903 -600   773  
2050-01-01 04:00:00 -950  700 -411  540  460 -700 -1000 -937 -600   229  

[5 rows x 71 columns]

In [71]:
# Put everything as positive values
pos = trade_agg.clip(lower=0).copy()
neg_rev = (-trade_agg.clip(upper=0)).copy()

neg_rev.columns = pd.MultiIndex.from_arrays(
    [
        neg_rev.columns.get_level_values("Country to"),
        neg_rev.columns.get_level_values("Country from"),
    ],
    names=["Country from", "Country to"],
)

trade_directional = (
    pd.concat([pos, neg_rev], axis=1)
    .T.groupby(level=["Country from", "Country to"])
    .sum()
    .T.swaplevel(axis=1)
)

In [72]:
# This allows the production of a bidirectional matrix for each hour
trade_directional.loc["2050-01-01 00:00:00"].unstack("Country from").fillna(0).head()

Country from,AL,AT,BA,BE,BG,CH,CY,CZ,DE,DK,...,NL,NO,PL,PT,RO,RS,SE,SI,SK,UK
Country to,,,,,,,,,,,,,,,,,,,,,
AL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AT,0.0,0.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,950.0,0.0,0.0
BA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1000.0
BG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Single matrix

In [78]:
# Harmonize names in column levels
# Source = ecoinvent/premise technology, or "country from"
# Country = country where the technology is located, or "country to"
prod_agg_ei.columns.names = trade_directional.columns.names = ["Country", "Source"]
trade_directional.xs("FR", level="Country", axis=1)

Source,BE,CH,DE,ES,IE,IT,LU,UK
2050-01-01 00:00:00,0,613,0,4553,700,2160,0,0
2050-01-01 01:00:00,0,0,0,1564,700,1608,0,0
2050-01-01 02:00:00,0,0,0,119,700,871,0,3
2050-01-01 03:00:00,0,0,0,0,700,2158,0,1034
2050-01-01 04:00:00,0,0,0,0,700,2158,0,2897


In [79]:
trade_directional.xs("FR", level="Source", axis=1)

Country,BE,CH,DE,ES,IE,IT,LU,UK
2050-01-01 00:00:00,4300,0,4800,0,0,0,121,1847
2050-01-01 01:00:00,4300,0,4800,0,0,0,134,174
2050-01-01 02:00:00,4300,0,4800,0,0,0,125,0
2050-01-01 03:00:00,4300,0,4800,1902,0,0,130,0
2050-01-01 04:00:00,4300,0,4800,3346,0,0,113,0


In [80]:
Z_gross = pd.concat([prod_agg_ei, trade_directional], axis=1).head()

In [81]:
Z_gross.loc["2050-01-01 00:00:00", "FR"].nlargest(25)

Source
electricity production, nuclear, boiling water reactor                                                            11537.0
electricity production, nuclear, pressure water reactor                                                           11537.0
electricity production, wind, <1MW turbine, onshore                                                                5776.0
electricity production, wind, >3MW turbine, onshore                                                                5776.0
electricity production, hydro, reservoir, alpine region                                                            4961.0
electricity production, hydro, reservoir, non-alpine region                                                        4961.0
electricity production, wind, 1-3MW turbine, offshore                                                              4721.0
electricity production, wind, 1-3MW turbine, onshore                                                               4721.0
ES               